### Coleta de dados de Notificação de casos de Dengue

Fonte: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip


In [ ]:
import sys, os, glob
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\DataSUS\\Dengue"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
print(os.path.basename('C:\\Marco Conti\\Projetos\\Dados\\DataSUS\\Dengue\\DENGBR14.csv')[6:8])  # Saída: '14'

In [ ]:
# Reordenar as colunas para manter a consistência entre os arquivos

def reorder_columns(df_):

    query = \
        """ Select TP_NOT
                ,ID_AGRAVO
                ,DT_NOTIFIC
                ,SEM_NOT
                ,NU_ANO
                ,SG_UF_NOT
                ,ID_MUNICIP
                ,ID_REGIONA
                ,ID_UNIDADE
                ,DT_SIN_PRI
                ,SEM_PRI
                ,ANO_NASC
                ,NU_IDADE_N
                ,CS_SEXO
                ,CS_GESTANT
                ,CS_RACA
                ,CS_ESCOL_N
                ,SG_UF
                ,ID_MN_RESI
                ,ID_RG_RESI
                ,ID_PAIS
                ,DT_INVEST
                ,ID_OCUPA_N
                ,FEBRE
                ,MIALGIA
                ,CEFALEIA
                ,EXANTEMA
                ,VOMITO
                ,NAUSEA
                ,DOR_COSTAS
                ,CONJUNTVIT
                ,ARTRITE
                ,ARTRALGIA
                ,PETEQUIA_N
                ,LEUCOPENIA
                ,LACO
                ,DOR_RETRO
                ,DIABETES
                ,HEMATOLOG
                ,HEPATOPAT
                ,RENAL
                ,HIPERTENSA
                ,ACIDO_PEPT
                ,AUTO_IMUNE
                ,DT_CHIK_S1
                ,DT_CHIK_S2
                ,DT_PRNT
                ,RES_CHIKS1
                ,RES_CHIKS2
                ,RESUL_PRNT
                ,DT_SORO
                ,RESUL_SORO
                ,DT_NS1
                ,RESUL_NS1
                ,DT_VIRAL
                ,RESUL_VI_N
                ,DT_PCR
                ,RESUL_PCR_
                ,SOROTIPO
                ,HISTOPA_N
                ,IMUNOH_N
                ,HOSPITALIZ
                ,DT_INTERNA
                ,UF
                ,MUNICIPIO
                ,TPAUTOCTO
                ,COUFINF
                ,COPAISINF
                ,COMUNINF
                ,CLASSI_FIN
                ,CRITERIO
                ,DOENCA_TRA
                ,CLINC_CHIK
                ,EVOLUCAO
                ,DT_OBITO
                ,DT_ENCERRA
                ,ALRM_HIPOT
                ,ALRM_PLAQ
                ,ALRM_VOM
                ,ALRM_SANG
                ,ALRM_HEMAT
                ,ALRM_ABDOM
                ,ALRM_LETAR
                ,ALRM_HEPAT
                ,ALRM_LIQ
                ,DT_ALRM
                ,GRAV_PULSO
                ,GRAV_CONV
                ,GRAV_ENCH
                ,GRAV_INSUF
                ,GRAV_TAQUI
                ,GRAV_EXTRE
                ,GRAV_HIPOT
                ,GRAV_HEMAT
                ,GRAV_MELEN
                ,GRAV_METRO
                ,GRAV_SANG
                ,GRAV_AST
                ,GRAV_MIOC
                ,GRAV_CONSC
                ,GRAV_ORGAO
                ,DT_GRAV
                ,MANI_HEMOR
                ,EPISTAXE
                ,GENGIVO
                ,METRO
                ,PETEQUIAS
                ,HEMATURA
                ,SANGRAM
                ,LACO_N
                ,PLASMATICO
                ,EVIDENCIA
                ,PLAQ_MENOR
                ,CON_FHD
                ,COMPLICA
                ,TP_SISTEMA
                ,NDUPLIC_N
                ,DT_DIGITA
                ,CS_FLXRET
                ,FLXRECEBI
                ,MIGRADO_W
            from temp_table 
        """
    df_.createOrReplaceTempView("temp_table")
    df_final = spark.sql(query)
    return df_final


In [ ]:
# 1. Pastas de origem e destino 
pasta_origem = Path(r"C:\Marco Conti\Projetos\Dados\DataSUS\Dengue")

# 3. Busca todos os arquivos CSV
arquivos_csv = glob.glob(str(pasta_origem / "*.csv"))
print(len(arquivos_csv), "arquivos_csv encontrados.")

arquivos_csv_filtered = [arq for arq in arquivos_csv if os.path.basename(arq)[6:8] in ('15','16','17','18','19')]
# print(f"Convertendo {len(arquivos_csv_filtered)} arquivos...")


for i, arq in enumerate(arquivos_csv_filtered):
    print(f"Verificando arquivo: {arq}")

    file_name = os.path.basename(arq)

    df_ = spark.read.csv(arq, header=True, inferSchema=False)
    df_columns = df_.columns
    if "DT_NASC" in df_columns and "DT_NASCIMENTO" not in df_columns:
        df_ = df_.withColumn("ANO_NASC", F.year(F.to_date(F.col("DT_NASC"), "yyyy-MM-dd")).cast("string")) 
        df_.drop("DT_NASC")

    if "DT_DIGITA" not in df_columns:
        df_ = df_.withColumn("DT_DIGITA", F.lit(None).cast("string"))

    if "MIGRADO_W" not in df_columns:
        df_ = df_.withColumn("MIGRADO_W", F.lit(None).cast("string"))
        
    df_final = reorder_columns(df_)
    file_name_new = file_name.replace('.csv', '_new_columns.csv')
    df_pandas_csv = df_final.toPandas().to_csv(r"C:\Marco Conti\Projetos\Dados\DataSUS\Dengue\{0}".format(file_name_new), index=False)
    


In [ ]:
# df_.select("DT_DIGITA","ANO_NASC").show(5)
# df_final.printSchema()
df_final.show(5)

In [ ]:
df_pandas_csv = df_final.toPandas().to_csv(r"C:\Marco Conti\Projetos\Dados\DataSUS\Dengue\Dengue_teste.csv", index=False)


In [ ]:
df_pandas